In [1]:
import pandas as pd

df = pd.read_csv('../warehouse/nyc_sales_all.csv')
print(df.head())
print(df.columns)

         borough neighborhood                       building_class_category  \
0  Staten Island     ANNADALE  01  ONE FAMILY HOMES                           
1  Staten Island     ANNADALE  01  ONE FAMILY HOMES                           
2  Staten Island     ANNADALE  01  ONE FAMILY HOMES                           
3  Staten Island     ANNADALE  01  ONE FAMILY HOMES                           
4  Staten Island     ANNADALE  01  ONE FAMILY HOMES                           

                                     address  zip_code  residential_units  \
0  4732 AMBOY ROAD                             10312.0                1.0   
1  4730 AMBOY ROAD                             10312.0                1.0   
2  4718 AMBOY ROAD                             10312.0                1.0   
3  4716 AMBOY ROAD                             10312.0                1.0   
4  4714 AMBOY ROAD                             10312.0                1.0   

   commercial_units  total_units  year_built  sale_price   sal

In [2]:
zip_summary = (
    df.groupby('zip_code')['sale_price']
      .agg(['mean', 'median', 'count'])
      .reset_index()
)

zip_summary.columns = [
    'zip_code',
    'avg_sale_price',
    'median_sale_price',
    'num_sales'
]

zip_summary.to_csv('../warehouse/zip_summary_2014_2024.csv', index=False)
print("Saved zip_summary_2014_2024.csv!")

Saved zip_summary_2014_2024.csv!


In [3]:
import pandas as pd

df = pd.read_csv('../warehouse/nyc_sales_all.csv')

# ensure datetime
df['sale_date'] = pd.to_datetime(df['sale_date'])

# ensure ZIP formatting (not required for borough analysis but safe)
df['zip_code'] = df['zip_code'].astype(str).str.zfill(5)

# filter again for safety
df = df[df['sale_date'].dt.year >= 2014]

# extract year
df['year'] = df['sale_date'].dt.year

borough_summary = (
    df.groupby(['year', 'borough'])['sale_price']
      .median()
      .reset_index()
)

borough_summary.columns = ['year', 'borough', 'median_sale_price']

borough_summary.to_csv('../warehouse/borough_median_saleprice_2014_2024.csv', index=False)
print("Saved borough median summary!")

Saved borough median summary!


In [4]:
import pandas as pd

df = pd.read_csv('../warehouse/nyc_sales_all.csv')

# Convert date and extract year
df['sale_date'] = pd.to_datetime(df['sale_date'])
df['year'] = df['sale_date'].dt.year

# Only keep 2014–2024
df = df[df['year'] >= 2014]

# Median price by borough for 2014
borough_2014 = (
    df[df['year'] == 2014]
    .groupby('borough')['sale_price']
    .median()
    .reset_index()
)
borough_2014.columns = ['borough', 'median_2014']

# Median price by borough for 2024
borough_2024 = (
    df[df['year'] == 2024]
    .groupby('borough')['sale_price']
    .median()
    .reset_index()
)
borough_2024.columns = ['borough', 'median_2024']

# Merge them
borough_change = borough_2014.merge(borough_2024, on='borough', how='inner')

# Percent change formula
borough_change['pct_change'] = (
    (borough_change['median_2024'] - borough_change['median_2014'])
    / borough_change['median_2014']
)

# (Optional) add a percentage column for easier Tableau formatting
borough_change['pct_change_percent'] = borough_change['pct_change'] * 100

# Save output
borough_change.to_csv('../warehouse/borough_pct_change_2014_2024.csv', index=False)
print("Saved borough_pct_change_2014_2024.csv! ✅")


Saved borough_pct_change_2014_2024.csv! ✅
